In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Libraries ready')

In [ ]:
DATA_PATH  = './new_data_2015_2026.xlsx'
SHEET_NAME = 'Monthly_Data'
MODEL_DIR  = './models'
OUTPUT_DIR = './outputs'
TARGETS    = ['Sales', 'Profit']

os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1 · Load Data & Auto-Detect Latest Month

In [ ]:
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df_raw['Discount Band'] = df_raw['Discount Band'].fillna('None')

# Auto-detect latest month in data
LATEST_YEAR  = df_raw['Year'].max()
LATEST_MONTH = df_raw[df_raw['Year'] == LATEST_YEAR]['Month Number'].max()
latest_date  = pd.Timestamp(f'{LATEST_YEAR}-{str(LATEST_MONTH).zfill(2)}-01')
next_date    = latest_date + pd.DateOffset(months=1)
NEXT_YEAR    = next_date.year
NEXT_MONTH   = next_date.month

SEASON_NAME = {12:'Winter',1:'Winter',2:'Winter',
               3:'Spring',4:'Spring',5:'Spring',
               6:'Summer',7:'Summer',8:'Summer',
               9:'Autumn',10:'Autumn',11:'Autumn'}

print(f'📦 Dataset       : {df_raw.shape[0]:,} rows · {df_raw["Product"].nunique()} products · {df_raw["Segment"].nunique()} segments')
print(f'📅 Latest data   : {latest_date.strftime("%B %Y")}  ({SEASON_NAME[LATEST_MONTH]})')
print(f'🔮 Predicting    : {next_date.strftime("%B %Y")}  ({SEASON_NAME[NEXT_MONTH]})')
print(f'\nProducts in data : {sorted(df_raw["Product"].unique())}')
print(f'Segments         : {sorted(df_raw["Segment"].unique())}')

## 2 · Encoding & Product Metadata

In [ ]:
SEASON_ENC = {'Spring':0,'Summer':1,'Autumn':2,'Winter':3}
DISC_ENC   = {'None':0,'Low':1,'Medium':2,'High':3}
DISC_RATE  = {'None':0.0,'Low':0.03,'Medium':0.05,'High':0.10}
SEG_ENC    = {s:i for i,s in enumerate(sorted(df_raw['Segment'].unique()))}
PROD_MAP   = {p:i for i,p in enumerate(sorted(df_raw['Product'].unique()))}
BASE_YEAR  = df_raw['Year'].min()

# Which months each product is active (learned from data)
PRODUCT_ACTIVE_MONTHS = (
    df_raw.groupby('Product')['Month Number']
    .apply(lambda x: sorted(x.unique().tolist()))
    .to_dict()
)

# Last known specs per product (price, mfg cost, segment)
PRODUCT_SPECS = (
    df_raw.sort_values(['Year','Month Number'])
    .groupby('Product')
    .last()[['Manufacturing Price','Sale Price','Segment']]
    .to_dict('index')
)

print(f'\nSegment encoding : {SEG_ENC}')
print(f'\nProducts active in {next_date.strftime("%B")}:')
active_next = [p for p,m in PRODUCT_ACTIVE_MONTHS.items() if NEXT_MONTH in m]
for p in sorted(active_next):
    spec = PRODUCT_SPECS[p]
    print(f'  {p:<22} Segment: {spec["Segment"]:<16} Price: {spec["Sale Price"]}  Mfg: {spec["Manufacturing Price"]}')

## 3 · Feature Engineering

In [ ]:
def build_features(df_input: pd.DataFrame) -> pd.DataFrame:
    df = df_input.copy()
    df['Discount Band'] = df['Discount Band'].fillna('None')
    df['Date']        = pd.to_datetime(df['Year'].astype(str)+'-'+df['Month Number'].astype(str).str.zfill(2)+'-01')
    df = df.sort_values(['Product','Date']).reset_index(drop=True)

    # Temporal
    df['Month']       = df['Date'].dt.month
    df['Year_num']    = df['Date'].dt.year
    df['Quarter']     = df['Date'].dt.quarter
    df['Month_sin']   = np.sin(2*np.pi*df['Month']/12)
    df['Month_cos']   = np.cos(2*np.pi*df['Month']/12)
    df['Qtr_sin']     = np.sin(2*np.pi*df['Quarter']/4)
    df['Qtr_cos']     = np.cos(2*np.pi*df['Quarter']/4)
    df['Season_enc']  = df['Month'].map(SEASON_NAME).map(SEASON_ENC)
    df['Year_trend']  = df['Year_num'] - BASE_YEAR

    # Categorical encodings
    df['Segment_enc']  = df['Segment'].map(SEG_ENC)
    df['Product_enc']  = df['Product'].map(PROD_MAP)
    df['DiscBand_enc'] = df['Discount Band'].map(DISC_ENC).fillna(0).astype(int)

    # Lag features (1,2,3 = short-term trend; 6 = half-year; 12 = same month last year)
    for lag in [1,2,3,6,12]:
        df[f'Sales_lag{lag}']  = df.groupby('Product')['Sales'].shift(lag)
        df[f'Profit_lag{lag}'] = df.groupby('Product')['Profit'].shift(lag)

    # Rolling stats (shift by 1 to avoid leakage)
    for win in [3,6]:
        df[f'Sales_roll{win}_mean']  = df.groupby('Product')['Sales'].transform(
            lambda x: x.shift(1).rolling(win, min_periods=1).mean())
        df[f'Sales_roll{win}_std']   = df.groupby('Product')['Sales'].transform(
            lambda x: x.shift(1).rolling(win, min_periods=2).std().fillna(0))
        df[f'Profit_roll{win}_mean'] = df.groupby('Product')['Profit'].transform(
            lambda x: x.shift(1).rolling(win, min_periods=1).mean())
        df[f'Profit_roll{win}_std']  = df.groupby('Product')['Profit'].transform(
            lambda x: x.shift(1).rolling(win, min_periods=2).std().fillna(0))

    # Financial derived features
    df['Discount_Rate']      = np.where(df['Gross Sales']!=0, df['Discounts']/df['Gross Sales'], 0)
    df['Revenue_per_Unit']   = np.where(df['Units Sold']!=0,  df['Sales']/df['Units Sold'],      0)
    df['Cost_per_Unit']      = np.where(df['Units Sold']!=0,  df['COGS']/df['Units Sold'],       0)
    df['Price_to_ManufCost'] = np.where(df['Manufacturing Price']!=0,
                                        df['Sale Price']/df['Manufacturing Price'], 0)
    return df


df = build_features(df_raw)

DROP_COLS    = ['Sales','Profit','Date','Segment','Product','Discount Band','Month Number','Year']
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
print(f'✅ Features engineered: {len(FEATURE_COLS)} features')
print(FEATURE_COLS)

## 4 · Train / Test Split & Model Training

In [ ]:
lag_cols  = [c for c in FEATURE_COLS if 'lag' in c or 'roll' in c]
df_model  = df.dropna(subset=lag_cols).reset_index(drop=True)
split_idx = int(len(df_model) * 0.80)
df_train  = df_model.iloc[:split_idx]
df_test   = df_model.iloc[split_idx:]
X_train   = df_train[FEATURE_COLS]
X_test    = df_test[FEATURE_COLS]

print(f'Training rows : {len(df_train):,}')
print(f'Test rows     : {len(df_test):,}')
print(f'Features      : {len(FEATURE_COLS)}')

In [ ]:
models, all_metrics = {}, {}

for tgt in TARGETS:
    y_tr = df_train[tgt]
    y_te = df_test[tgt]

    model = HistGradientBoostingRegressor(
        max_iter            = 600,
        learning_rate       = 0.04,
        max_depth           = 6,
        min_samples_leaf    = 10,
        l2_regularization   = 0.1,
        max_leaf_nodes      = 40,
        early_stopping      = True,
        validation_fraction = 0.15,
        n_iter_no_change    = 35,
        random_state        = 42,
        verbose             = 0
    )
    model.fit(X_train, y_tr)

    preds = model.predict(X_test)
    r2    = r2_score(y_te, preds)
    rmse  = np.sqrt(mean_squared_error(y_te, preds))
    mae   = mean_absolute_error(y_te, preds)
    mape  = np.mean(np.abs((y_te - preds) / np.where(y_te==0, 1, y_te))) * 100

    print(f'\n  {'═'*48}')
    print(f'  🎯  {tgt.upper()}  — TEST METRICS')
    print(f'  {'═'*48}')
    print(f'  R²   : {r2:.4f}   ({"🟢 Excellent" if r2>0.95 else "🟡 Good" if r2>0.85 else "🔴 Review"})')
    print(f'  MAPE : {mape:.2f} %')
    print(f'  RMSE : {rmse:>12,.0f}')
    print(f'  MAE  : {mae:>12,.0f}')

    joblib.dump(model, f'{MODEL_DIR}/{tgt.lower()}_model.pkl')
    models[tgt]      = model
    all_metrics[tgt] = dict(r2=r2, mape=mape, rmse=rmse, mae=mae,
                             preds=preds, y_test=y_te.values)

# Persist all metadata needed for inference
joblib.dump(FEATURE_COLS,           f'{MODEL_DIR}/feature_cols.pkl')
joblib.dump(PROD_MAP,               f'{MODEL_DIR}/prod_map.pkl')
joblib.dump(SEG_ENC,                f'{MODEL_DIR}/seg_enc.pkl')
joblib.dump(PRODUCT_ACTIVE_MONTHS,  f'{MODEL_DIR}/active_months.pkl')
joblib.dump(PRODUCT_SPECS,          f'{MODEL_DIR}/product_specs.pkl')
joblib.dump(BASE_YEAR,              f'{MODEL_DIR}/base_year.pkl')
print(f'\n💾 Models + metadata saved to {MODEL_DIR}/')

## 5 · Visualisations

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, tgt in zip(axes, TARGETS):
    met = all_metrics[tgt]
    ax.plot(met['y_test'], label='Actual',    lw=1.5, color='steelblue')
    ax.plot(met['preds'],  label='Predicted', lw=1.5, color='tomato', ls='--')
    ax.set_title(f'{tgt} — Actual vs Predicted (Test Set)', fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e3:,.0f}K'))
    ax.legend()
    ax.text(0.02, 0.97, f'R²={met["r2"]:.4f}  MAPE={met["mape"]:.1f}%',
            transform=ax.transAxes, va='top', fontsize=10,
            bbox=dict(boxstyle='round', alpha=0.2))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
# Seasonality profile per segment
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
segments    = sorted(df_raw['Segment'].unique())
fig, axes   = plt.subplots(2, 3, figsize=(18, 10))

for ax, seg in zip(axes.flat, segments):
    seg_df = df_raw[df_raw['Segment']==seg]
    monthly = seg_df.groupby('Month Number')['Sales'].mean().reindex(range(1,13), fill_value=0)
    colors  = ['#2196F3' if SEASON_NAME[m]=='Winter' else '#4CAF50' if SEASON_NAME[m]=='Spring'
               else '#FF9800' if SEASON_NAME[m]=='Summer' else '#9C27B0' for m in range(1,13)]
    ax.bar(month_names, monthly.values, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_title(f'{seg}', fontweight='bold', fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e3:,.0f}K'))
    ax.set_xticklabels(month_names, fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(color='#2196F3',label='Winter'), Patch(color='#4CAF50',label='Spring'),
              Patch(color='#FF9800',label='Summer'), Patch(color='#9C27B0',label='Autumn')]
fig.legend(handles=legend_els, loc='lower center', ncol=4, fontsize=11, framealpha=0.8)
fig.suptitle('Average Monthly Sales by Segment — Seasonal Profile', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/seasonal_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Seasonal chart saved.')

In [ ]:
# Annual Sales trend per product (top 12)
top_prods = df_raw.groupby('Product')['Sales'].sum().nlargest(12).index.tolist()
annual    = df_raw[df_raw['Product'].isin(top_prods)].groupby(['Product','Year'])['Sales'].sum().reset_index()

fig, ax = plt.subplots(figsize=(16, 6))
for prod in top_prods:
    d = annual[annual['Product']==prod]
    ax.plot(d['Year'], d['Sales']/1e6, marker='o', markersize=4, label=prod, linewidth=1.5)
ax.set_title('Annual Sales Trend — Top 12 Products (2015–2026)', fontweight='bold', fontsize=13)
ax.set_xlabel('Year'); ax.set_ylabel('Sales (Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.1f}M'))
ax.legend(loc='upper left', ncol=2, fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/annual_trend_top_products.png', dpi=150)
plt.show()
print('📈 Trend chart saved.')

## 6 · Prediction Engine

In [ ]:
def predict_product(
    target_year:   int,
    target_month:  int,
    product:       str,
    discount_band: str   = None,
    units_sold:    float = None,
    extra_data:    pd.DataFrame = None,
) -> dict:
    """
    Predict Sales & Profit for one product in one month.

    Parameters
    ----------
    target_year   : Year to predict
    target_month  : Month 1–12
    product       : Product name (must exist in training data)
    discount_band : 'None'|'Low'|'Medium'|'High'  (uses product's mode if None)
    units_sold    : Expected units  (uses product's same-month avg if None)
    extra_data    : DataFrame of new months (same schema as Monthly_Data sheet)
                    New data takes priority over old training data for lags!
    """
    # Combine old data + any new data
    if extra_data is not None:
        nd = extra_data.copy()
        if 'Month Number' not in nd.columns and 'Month' in nd.columns:
            nd = nd.rename(columns={'Month':'Month Number'})
        combined = pd.concat([df_raw, nd], ignore_index=True).drop_duplicates(
            subset=['Product','Year','Month Number'], keep='last')
    else:
        combined = df_raw.copy()

    hist = combined[combined['Product']==product].copy()
    hist['Date'] = pd.to_datetime(
        hist['Year'].astype(str)+'-'+hist['Month Number'].astype(str).str.zfill(2)+'-01')
    hist = hist.sort_values('Date').reset_index(drop=True)

    spec  = PRODUCT_SPECS[product]
    mfg   = spec['Manufacturing Price']
    sprice= spec['Sale Price']
    seg   = spec['Segment']

    if discount_band is None:
        discount_band = hist['Discount Band'].mode()[0] if not hist.empty else 'None'
    if units_sold is None:
        same_mo   = hist[hist['Month Number']==target_month]
        units_sold= same_mo['Units Sold'].mean() if not same_mo.empty else hist['Units Sold'].mean()

    month       = target_month
    quarter     = (month-1)//3+1
    target_date = pd.Timestamp(f'{target_year}-{str(month).zfill(2)}-01')

    # ── Resolve lag values (new data first, then old, then seasonal avg) ─
    def get_lag(col, lag):
        cutoff = target_date - pd.DateOffset(months=lag)
        row    = hist[hist['Date']==cutoff]
        if not row.empty: return row[col].values[0]
        same_m = hist[hist['Month Number']==cutoff.month]
        return same_m[col].mean() if not same_m.empty else hist[col].mean()

    sl    = {f'Sales_lag{l}':  get_lag('Sales',  l) for l in [1,2,3,6,12]}
    pl    = {f'Profit_lag{l}': get_lag('Profit', l) for l in [1,2,3,6,12]}
    sv,pv = hist['Sales'].values, hist['Profit'].values
    rolls = {
        'Sales_roll3_mean':  np.mean(sv[-3:]) if len(sv)>=3 else np.mean(sv),
        'Sales_roll3_std':   np.std(sv[-3:])  if len(sv)>=3 else 0,
        'Sales_roll6_mean':  np.mean(sv[-6:]) if len(sv)>=6 else np.mean(sv),
        'Sales_roll6_std':   np.std(sv[-6:])  if len(sv)>=6 else 0,
        'Profit_roll3_mean': np.mean(pv[-3:]) if len(pv)>=3 else np.mean(pv),
        'Profit_roll3_std':  np.std(pv[-3:])  if len(pv)>=3 else 0,
        'Profit_roll6_mean': np.mean(pv[-6:]) if len(pv)>=6 else np.mean(pv),
        'Profit_roll6_std':  np.std(pv[-6:])  if len(pv)>=6 else 0,
    }

    dr = DISC_RATE.get(discount_band, 0)
    eg = units_sold * sprice
    row_dict = {
        'Units Sold':units_sold,'Manufacturing Price':mfg,'Sale Price':sprice,
        'Gross Sales':eg,'Discounts':eg*dr,'COGS':units_sold*mfg,
        'Month':month,'Year_num':target_year,'Quarter':quarter,
        'Month_sin':np.sin(2*np.pi*month/12),'Month_cos':np.cos(2*np.pi*month/12),
        'Qtr_sin':np.sin(2*np.pi*quarter/4),'Qtr_cos':np.cos(2*np.pi*quarter/4),
        'Season_enc':SEASON_ENC[SEASON_NAME[month]],
        'Year_trend':target_year-BASE_YEAR,
        'Segment_enc':SEG_ENC.get(seg,0),
        'Product_enc':PROD_MAP.get(product,0),
        'DiscBand_enc':DISC_ENC.get(discount_band,0),
        'Discount_Rate':dr,'Revenue_per_Unit':sprice*(1-dr),
        'Cost_per_Unit':mfg,'Price_to_ManufCost':sprice/mfg if mfg else 0,
        **sl,**pl,**rolls,
    }

    X = pd.DataFrame([row_dict])[FEATURE_COLS]
    return {
        'Product':          product,
        'Segment':          seg,
        'Predicting':       target_date.strftime('%B %Y'),
        'Season':           SEASON_NAME[month],
        'Discount_Band':    discount_band,
        'Units_Sold_Est':   round(units_sold, 1),
        'Predicted_Sales':  round(models['Sales'].predict(X)[0],  2),
        'Predicted_Profit': round(models['Profit'].predict(X)[0], 2),
    }


print('✅ predict_product() ready')

## 7 · Predict Next Month (Auto)

In [ ]:
#  AUTO-PREDICTION — Next month based on latest data in your file

print(f'📅 Latest data in file : {latest_date.strftime("%B %Y")}  ({SEASON_NAME[LATEST_MONTH]})')
print(f'🔮 Predicting          : {next_date.strftime("%B %Y")}  ({SEASON_NAME[NEXT_MONTH]})')
print(f'🛒 Active products     : {len([p for p,m in PRODUCT_ACTIVE_MONTHS.items() if NEXT_MONTH in m])}')
print()

auto_results = []
for prod in sorted(PRODUCT_ACTIVE_MONTHS.keys()):
    if NEXT_MONTH not in PRODUCT_ACTIVE_MONTHS[prod]:
        continue      # skip products not active this season
    r = predict_product(NEXT_YEAR, NEXT_MONTH, prod)
    auto_results.append(r)

df_auto = pd.DataFrame(auto_results)

print(f'{'═'*75}')
print(f'  🗓  PREDICTIONS FOR {next_date.strftime("%B %Y").upper()}  ·  Season: {SEASON_NAME[NEXT_MONTH]}')
print(f'{'═'*75}')
print(f'  {'Product':<22} {'Segment':<16} {'Sales':>16} {'Profit':>16} {'Margin':>8}')
print(f'  {'-'*75}')

for _, r in df_auto.iterrows():
    margin = r['Predicted_Profit'] / r['Predicted_Sales'] * 100 if r['Predicted_Sales'] else 0
    print(f'  {r["Product"]:<22} {r["Segment"]:<16} {r["Predicted_Sales"]:>16,.0f} {r["Predicted_Profit"]:>16,.0f} {margin:>7.1f}%')

print(f'  {'─'*75}')
total_s = df_auto['Predicted_Sales'].sum()
total_p = df_auto['Predicted_Profit'].sum()
print(f'  {'TOTAL':<38} {total_s:>16,.0f} {total_p:>16,.0f} {total_p/total_s*100:>7.1f}%')
print(f'{'═'*75}')
display(df_auto)

In [ ]:
# Prediction bar chart
seg_colors = {'Personal Care':'#66BB6A','Home & Linen':'#42A5F5','Kitchen':'#FFA726',
              'Stationery':'#AB47BC','Health':'#EF5350','Seasonal':'#FF7043'}

df_plot = df_auto.sort_values('Predicted_Sales', ascending=True)
colors  = [seg_colors.get(s,'#90A4AE') for s in df_plot['Segment']]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, col, label in zip(axes,
                           ['Predicted_Sales','Predicted_Profit'],
                           ['Predicted Sales','Predicted Profit']):
    df_sorted = df_auto.sort_values(col, ascending=True)
    clrs = [seg_colors.get(s,'#90A4AE') for s in df_sorted['Segment']]
    bars = ax.barh(df_sorted['Product'], df_sorted[col], color=clrs,
                   edgecolor='white', linewidth=0.8, height=0.7)
    ax.set_title(f'{label} — {next_date.strftime("%B %Y")} ({SEASON_NAME[NEXT_MONTH]})',
                 fontweight='bold', fontsize=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e3:,.0f}K'))
    for bar, val in zip(bars, df_sorted[col]):
        ax.text(val + max(df_sorted[col])*0.01, bar.get_y()+bar.get_height()/2,
                f'{val/1e3:,.0f}K', va='center', fontsize=8)

from matplotlib.patches import Patch
legend_els = [Patch(color=c, label=s) for s,c in seg_colors.items()]
fig.legend(handles=legend_els, loc='lower center', ncol=6, fontsize=9, framealpha=0.9)
plt.tight_layout(rect=[0,0.06,1,1])
plt.savefig(f'{OUTPUT_DIR}/next_month_prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Prediction chart saved.')

## 8 · Predict Any Specific Month

In [ ]:
#  CHANGE THESE to predict any month
TARGET_YEAR   = 2026
TARGET_MONTH  = 4       

target_date_m = pd.Timestamp(f'{TARGET_YEAR}-{str(TARGET_MONTH).zfill(2)}-01')
active_prods  = [p for p,m in PRODUCT_ACTIVE_MONTHS.items() if TARGET_MONTH in m]

print(f'📅 Predicting: {target_date_m.strftime("%B %Y")}  ({SEASON_NAME[TARGET_MONTH]})')
print(f'🛒 Active products: {sorted(active_prods)}')

manual_results = []
for prod in sorted(active_prods):
    r = predict_product(TARGET_YEAR, TARGET_MONTH, prod)
    manual_results.append(r)

df_manual = pd.DataFrame(manual_results)
print(f'\nTotal predicted Sales  : {df_manual["Predicted_Sales"].sum():>15,.0f}')
print(f'Total predicted Profit : {df_manual["Predicted_Profit"].sum():>15,.0f}')
display(df_manual)

## 9 · Full Year Forecast for One Product

In [ ]:
#  CHANGE product and year
FORECAST_PRODUCT = 'Shampoo'
FORECAST_YEAR    = 2026

active_m = PRODUCT_ACTIVE_MONTHS[FORECAST_PRODUCT]
fy_rows  = []

for mo in range(1, 13):
    if mo not in active_m:
        fy_rows.append({'Month': pd.Timestamp(f'{FORECAST_YEAR}-{str(mo).zfill(2)}-01').strftime('%b'),
                        'Predicted_Sales':0,'Predicted_Profit':0,'Active':False})
    else:
        r = predict_product(FORECAST_YEAR, mo, FORECAST_PRODUCT)
        fy_rows.append({'Month': pd.Timestamp(f'{FORECAST_YEAR}-{str(mo).zfill(2)}-01').strftime('%b'),
                        'Predicted_Sales':r['Predicted_Sales'],
                        'Predicted_Profit':r['Predicted_Profit'],
                        'Active':True})

df_fy = pd.DataFrame(fy_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, clr in zip(axes, ['Predicted_Sales','Predicted_Profit'], ['steelblue','seagreen']):
    bar_colors = [clr if r['Active'] else '#E0E0E0' for _,r in df_fy.iterrows()]
    ax.bar(df_fy['Month'], df_fy[col], color=bar_colors, edgecolor='white', linewidth=0.8)
    ax.set_title(f'{FORECAST_PRODUCT} — {col.replace("_"," ")} {FORECAST_YEAR}',
                 fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e3:,.0f}K'))
    ax.text(0.98, 0.97, f'Active months: {active_m}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, bbox=dict(boxstyle='round', alpha=0.2))
    for bar, val in zip(ax.patches, df_fy[col]):
        if val > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(df_fy[col])*0.01,
                    f'{val/1e3:.0f}K', ha='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/{FORECAST_PRODUCT}_full_year_{FORECAST_YEAR}.png', dpi=150)
plt.show()
print(f'📊 Full-year forecast chart saved.')
display(df_fy[df_fy['Active']])

## 10 · Export All Results to Excel

In [ ]:
out_path = f'{OUTPUT_DIR}/forecast_results.xlsx'

# Build a full year forecast for ALL products
all_year_rows = []
for prod in sorted(PRODUCT_ACTIVE_MONTHS.keys()):
    for mo in range(1, 13):
        if mo in PRODUCT_ACTIVE_MONTHS[prod]:
            r = predict_product(FORECAST_YEAR, mo, prod)
            r['Month_Num'] = mo
            all_year_rows.append(r)
df_all_year = pd.DataFrame(all_year_rows)

# Pivot summary
pivot_sales  = df_all_year.pivot_table(index='Product', columns='Predicting',
                                        values='Predicted_Sales', aggfunc='sum').fillna(0)
pivot_profit = df_all_year.pivot_table(index='Product', columns='Predicting',
                                        values='Predicted_Profit', aggfunc='sum').fillna(0)

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    df_auto.to_excel(writer,      sheet_name=f'{next_date.strftime("%b_%Y")}_AutoPredict', index=False)
    df_manual.to_excel(writer,    sheet_name=f'{target_date_m.strftime("%b_%Y")}_Manual',  index=False)
    df_all_year.to_excel(writer,  sheet_name='FullYear_AllProducts', index=False)
    pivot_sales.to_excel(writer,  sheet_name='Pivot_Sales')
    pivot_profit.to_excel(writer, sheet_name='Pivot_Profit')
    pd.DataFrame([{'Target':t,'R2':round(all_metrics[t]['r2'],4),
                   'MAPE_%':round(all_metrics[t]['mape'],2),
                   'RMSE':round(all_metrics[t]['rmse'],0),
                   'MAE':round(all_metrics[t]['mae'],0)} for t in TARGETS]
    ).to_excel(writer, sheet_name='Model_Metrics', index=False)

print(f'✅ Results exported → {out_path}')
print(f'   Sheets: Auto-predict · Manual predict · Full year all products · Pivot Sales · Pivot Profit · Metrics')